# Step 6: Evaluation
A frozen 30 question test set (20 grounded questions whose answers were checked against their source passage, 10 edge cases). The judge is a stronger model than the answerer, at temperature 0, with a rubric. Deterministic checks sit beside it. All runs are tracked in MLflow.

In [1]:
import sys, json, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.max_colwidth', 120)

In [2]:
ts = json.load(open('../reports/testset.json'))
pd.DataFrame(ts['items'])[['id', 'type', 'question', 'expect']]

,id,type,question,expect
0,G1,factual,"According to Royal Bank of Canada's fiscal 2025 Annual Report, what was the total fair value of securities with liqu...",answer
1,G2,factual,"According to Bank of Nova Scotia's fiscal 2024 Annual Report (MD&A), what was the after-tax impact of the impairment...",answer
2,G3,factual,"What was Bank of Montreal's Common Equity Tier 1 (CET1) Ratio as at October 31, 2024, according to its fiscal 2024 A...",answer
3,G4,factual,"According to Royal Bank of Canada's Annual Information Form for fiscal 2025, what is the maximum aggregate considera...",answer
4,G5,factual,"According to Bank of Nova Scotia's fiscal 2024 Annual Report, what was the Bank's total risk-weighted assets (RWA) a...",answer
5,G6,factual,"According to CIBC's fiscal 2024 Annual Report (MD&A), what were total assets as at October 31, 2024?",answer
6,G7,factual,How many full-time equivalent employees did Toronto-Dominion Bank have on average in fiscal 2025?,answer
7,G8,factual,"According to Toronto-Dominion Bank's fiscal 2025 annual report, what was the total amount of deposits under the Schw...",answer
8,G9,factual,"According to CIBC's fiscal 2024 Annual Information Form, what is the maximum aggregate consideration for all outstan...",answer
9,G10,factual,"According to Royal Bank of Canada's fiscal 2024 annual report, what was the total pension and other post-employment ...",answer


In [3]:
ev = json.load(open('../reports/evaluation_results.json'))
table = pd.DataFrame([{k: v[k] for k in ('config', 'faithfulness', 'relevance', 'completeness', 'key_figures_present', 'figures_supported', 'edge_behaviour_ok', 'usd_per_query', 'median_ms')} for v in ev.values()])
table

,config,faithfulness,relevance,completeness,key_figures_present,figures_supported,edge_behaviour_ok,usd_per_query,median_ms
0,basic_topk,4.500,4.633,4.433,0.50,0.974,0.778,0.004,1890.5
1,mmr_structured,4.571,4.571,4.536,0.55,0.938,0.889,0.004,1953.0
2,best,4.900,4.733,4.633,0.60,0.971,0.889,0.004,8563.0
3,agent,4.700,4.867,4.667,0.80,0.841,1.000,0.009,9359.0


In [4]:
for name, v in ev.items():
    print(name, 'hallucinations (faithfulness <= 2):', v['hallucinations'])

basic_topk hallucinations (faithfulness <= 2): ['G5', 'G18', 'E3', 'E6']
mmr_structured hallucinations (faithfulness <= 2): ['G18', 'E6']
best hallucinations (faithfulness <= 2): ['E6']
agent hallucinations (faithfulness <= 2): ['G7', 'E10']


In [5]:
cost = table.set_index('config').usd_per_query
(cost * 1000 * 365).rename('USD per year at 1,000 queries a day')

config
basic_topk        1460.0
mmr_structured    1460.0
best              1460.0
agent             3285.0
Name: USD per year at 1,000 queries a day, dtype: float64

## Set B after the quarterly reports were added, and on OpenAI
The first run after the change found a scope regression (one question sent to the Q1 reports by a date); after the fix two identical runs differ by about one item in twenty, which is the run to run spread. The OpenAI run (gpt-4o-mini, judged by gpt-4o) uses a different judge, so it compares only roughly.

In [6]:
runs = {'set B, original': 'evaluation_results_b.json', 'after change, first run': 'evaluation_results_b_v3.json', 'after fix, run 1': 'evaluation_results_b_v3_run1.json', 'after fix, run 2': 'evaluation_results_b_v3_run2.json', 'OpenAI': 'evaluation_results_b_openai.json'}
pd.DataFrame([{'run': n, **{k: json.load(open('../reports/' + f))['best_v2'][k] for k in ('faithfulness', 'relevance', 'completeness', 'key_figures_present', 'edge_behaviour_ok', 'usd_per_query')}} for n, f in runs.items()])

,run,faithfulness,relevance,completeness,key_figures_present,edge_behaviour_ok,usd_per_query
0,"set B, original",4.931,4.828,4.828,0.80,0.889,0.005
1,"after change, first run",5.000,4.893,4.893,0.65,0.889,0.004
2,"after fix, run 1",4.931,4.862,4.862,0.80,0.889,0.005
3,"after fix, run 2",4.929,4.857,4.857,0.75,0.889,0.005
4,OpenAI,4.733,4.867,4.733,0.75,0.778,0.001


In [7]:
rows = {n: {r['id']: r['key_figures_present'] for r in json.load(open('../reports/' + f))['best_v2']['rows'] if r['id'].startswith('G')} for n, f in runs.items() if n != 'OpenAI'}
k = pd.DataFrame(rows)
k[k.nunique(axis=1) > 1]

,"set B, original","after change, first run","after fix, run 1","after fix, run 2"
G2,True,False,True,False
G4,False,False,True,False
G6,True,False,False,True
G7,True,False,True,True


## The final design (v4, v5)
v4: neighbouring chunks, verified computations, the agent's survey and calculator tools, and the judge with its full context (see calibration below). v5 adds comparison searches in parts, the period check on table figures and the quote rule. Set A, on which none of these changes were made, is the cleanest test of v5.

In [8]:
v4 = {'v1 best, set A': ('evaluation_results.json', 'best'), 'v5, set A': ('evaluation_results_a_v5.json', 'best_v2'), 'v4 run 1, set B': ('evaluation_results_b_v4_run1.json', 'best_v2'), 'v4 run 2, set B': ('evaluation_results_b_v4_run2.json', 'best_v2'), 'v5, set B': ('evaluation_results_b_v5.json', 'best_v2'), 'agent v4, set B': ('evaluation_results_b_v4_agent.json', 'agent')}
pd.DataFrame([{'run': n, **{k: json.load(open('../reports/' + f))[c][k] for k in ('faithfulness', 'relevance', 'completeness', 'key_figures_present', 'edge_behaviour_ok', 'usd_per_query', 'median_ms')}} for n, (f, c) in v4.items()])

,run,faithfulness,relevance,completeness,key_figures_present,edge_behaviour_ok,usd_per_query,median_ms
0,"v1 best, set A",4.900,4.733,4.633,0.60,0.889,0.004,8563.0
1,"v5, set A",5.000,4.931,4.862,0.75,0.889,0.007,15359.5
2,"v4 run 1, set B",4.862,4.759,4.793,0.70,0.889,0.006,16797.5
3,"v4 run 2, set B",5.000,4.852,4.889,0.65,0.889,0.007,20461.5
4,"v5, set B",5.000,4.862,4.759,0.80,0.889,0.006,7672.0
5,"agent v4, set B",4.931,4.793,4.793,0.85,1.000,0.009,15422.0


## Judge calibration against human verdicts
The judge scored 60 Step 3 answers that a person had already graded against the hand located truth. The first calibration exposed a defect in the judge: it saw only the first 14,000 characters of context, so answers citing late passages looked unsupported. With the full context it agrees with the person on 95% of answers (Cohen's kappa 0.81).

In [9]:
cal = json.load(open('../reports/judge_calibration.json'))['summary']
old = json.load(open('../reports/judge_calibration_14k.json'))
print('before the fix (14,000 characters): agreement', round(sum(r['human_pass'] == r['judge_pass'] for r in old['rows']) / len(old['rows']), 2))
print('after: agreement', cal['agreement'], 'kappa', cal['kappa'])
pd.Series(cal['table'])

before the fix (14,000 characters): agreement 0.8
after: agreement 0.95 kappa 0.812


both pass                    49
both fail                     8
judge passes, human fails     1
judge fails, human passes     2
dtype: int64

In [10]:
pd.DataFrame(cal['disagreements'])[['id', 'config', 'human', 'faithfulness', 'relevance', 'completeness', 'reason']]

,id,config,human,faithfulness,relevance,completeness,reason
0,Q14,native:cot,partly correct,5,5,5,The answer accurately reflects what the context provides: it states the change in average total VaR ($0.4 million in...
1,Q07,native:cot+openai,correct,5,5,3,"The answer correctly states BMO's total PCL of $1,054 million in Q2 2025, which is directly supported by multiple co..."
2,Q11,native:cot+openai,correct,5,5,3,"The answer correctly states 131% for Q4 2024 (quarter ended October 31, 2024), which is supported by both S9 and S12..."
